# Lesson 2 - Vector Search - Code Along 3 - sqlitesearch

In this notebook, I will load the sqlitesearch index from the db file and use it
for RAG.

In [4]:
# dependencies
from dotenv import load_dotenv
from sqlitesearch import VectorSearchIndex
from sentence_transformers import SentenceTransformer
from anthropic import Anthropic

from rag_helper import RAGVector

In [5]:
# load environment variables
load_dotenv()

True

In [2]:
# load model for embedding new queries and index for search
model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# test a search
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'doc_id': '5ca6890c1a'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CO

In [6]:
# initialize LLM client
llm_client = Anthropic()

In [7]:
# initialize RAG assistant using the persistent vector search db
assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=llm_client,
)

In [ ]:
# try a search
# while index was made without filtering for a course, the RAG class has
# a filter for course, too, and llm-zoomcamp is the default value
reponse = assistant.rag("the program has already begun, can I still sign up?")
print(reponse[0])

# Can I Still Sign Up if the Program Has Already Begun?

**Yes, you can still join!** 

However, there's an important caveat: **If you want to receive a certificate, you need to submit your project while submissions are still being accepted.**

So while the course has started, you're welcome to begin learning and participating. Just make sure to complete and submit your Capstone project before the submission deadline closes to be eligible for the certificate.


In [13]:
# close db connection
vs_index.close()